In [1]:
# Import libaries
import pandas as pd
from sklearn.model_selection import train_test_split 
import time
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import chi2
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression
import pickle
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Functions
def selectkbest(indep_X,dep_Y,n):
    print("enter")
    test = SelectKBest(score_func=chi2, k=n)
    fit1= test.fit(indep_X,dep_Y)
    #summarize scores       
    selectk_features = fit1.transform(indep_X)
    return selectk_features
    
def split_scalar(indep_X,dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(indep_X, dep_Y, test_size = 0.25, random_state = 0)
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)    
    return X_train, X_test, y_train, y_test
    
def r2_prediction(regressor,X_test,y_test):
     y_pred = regressor.predict(X_test)
     from sklearn.metrics import r2_score
     r2=r2_score(y_test,y_pred)
     return r2
 
def Linear(X_train,y_train,X_test):       
    # Fitting K-NN to the Training set
    from sklearn.linear_model import LinearRegression
    regressor = LinearRegression()
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2   
    
def svm_linear(X_train,y_train,X_test):
                
    from sklearn.svm import SVR
    regressor = SVR(kernel = 'linear')
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2  
    
def svm_NL(X_train,y_train,X_test):
                
    from sklearn.svm import SVR
    regressor = SVR(kernel = 'rbf')
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2  
     

def Decision(X_train,y_train,X_test):
        
    # Fitting K-NN to the Training setC
    from sklearn.tree import DecisionTreeRegressor
    regressor = DecisionTreeRegressor(random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2  
     

def random(X_train,y_train,X_test):       
    # Fitting K-NN to the Training set
    from sklearn.ensemble import RandomForestRegressor
    regressor = RandomForestRegressor(n_estimators = 10, random_state = 0)
    regressor.fit(X_train, y_train)
    r2=r2_prediction(regressor,X_test,y_test)
    return  r2 

In [3]:
# Create DataFrame
def selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf): 
    
    dataframe=pd.DataFrame(index=['ChiSquare'],columns=['Linear','SVMl','SVMnl','Decision','Random'
                                                                                     ])

    for number,idex in enumerate(dataframe.index):
        
        dataframe['Linear'][idex]=acclin[number]       
        dataframe['SVMl'][idex]=accsvml[number]
        dataframe['SVMnl'][idex]=accsvmnl[number]
        dataframe['Decision'][idex]=accdes[number]
        dataframe['Random'][idex]=accrf[number]
    return dataframe

In [5]:
# Read dataset CSV File
dataset1=pd.read_csv("pre_processed_synthetic_ev.csv",index_col=None)
df2 = pd.get_dummies(dataset1, drop_first=True)
df2


,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
0,165,3.43,3.45,29.2,40.2,47.54,95.33
1,386,4.19,3.09,30.0,48.6,46.98,90.41
2,303,4.18,3.96,28.8,39.6,46.37,91.79
3,1572,4.02,1.46,22.8,92.2,33.49,69.18
4,1708,3.30,0.89,27.7,100.9,33.65,66.52
...,...,...,...,...,...,...,...
9995,1076,3.82,1.18,25.5,75.0,39.35,77.71
9996,398,3.80,3.36,26.0,43.0,46.16,92.31
9997,452,3.64,4.98,28.7,47.2,46.04,89.68
9998,116,4.12,3.12,27.9,30.5,48.63,96.10


In [6]:
df2.head()

,Cycles,Voltage_V,Current_A,Temperature_C,Internal_Resistance_mOhm,Capacity_Ah,SOH
0,165,3.43,3.45,29.2,40.2,47.54,95.33
1,386,4.19,3.09,30.0,48.6,46.98,90.41
2,303,4.18,3.96,28.8,39.6,46.37,91.79
3,1572,4.02,1.46,22.8,92.2,33.49,69.18
4,1708,3.30,0.89,27.7,100.9,33.65,66.52


In [7]:
df2 = pd.get_dummies(df2, drop_first=True)

indep_X=df2.drop('SOH', axis=1)
dep_Y=df2['SOH']

In [8]:
dep_Y

0       95.33
1       90.41
2       91.79
3       69.18
4       66.52
        ...  
9995    77.71
9996    92.31
9997    89.68
9998    96.10
9999    89.20
Name: SOH, Length: 10000, dtype: float64

In [9]:
kbest=selectkbest(indep_X,dep_Y,2)      

acclin=[]
accsvml=[]
accsvmnl=[]
accdes=[]
accrf=[]

X_train, X_test, y_train, y_test=split_scalar(kbest,dep_Y) 
X_train

enter


ValueError: Unknown label type: (array([95.33, 90.41, 91.79, ..., 89.68, 96.1 , 89.2 ]),)

In [ ]:
 
for i in kbest:   
    r2_lin=Linear(X_train,y_train,X_test)
    acclin.append(r2_lin)
    
    r2_sl=svm_linear(X_train,y_train,X_test)    
    accsvml.append(r2_sl)
    
    r2_NL=svm_NL(X_train,y_train,X_test)
    accsvmnl.append(r2_NL)
    
    r2_d=Decision(X_train,y_train,X_test)
    accdes.append(r2_d)
    
    r2_r=random(X_train,y_train,X_test)
    accrf.append(r2_r)
    
    
result=selectk_regression(acclin,accsvml,accsvmnl,accdes,accrf)

In [ ]:
# result k = 5
result